# RAG-Based Domain Tutor Chatbot
**NLP Course Tutor with Hallucination Detection**

Pipeline:
1. Load syllabus/textbook corpus
2. Chunk → Embed → Store in FAISS
3. Tutor-specific RAG chain (LCEL)
4. Hallucination detection (semantic faithfulness)
5. Interactive chat

## Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
print('API Key loaded:', 'Yes' if GROQ_API_KEY else 'No — check .env file')

API Key loaded: Yes


## Step 1: Load the Domain Corpus

In [2]:
from langchain_community.document_loaders import TextLoader

corpus_dir = './syllabus_docs'
documents = []

for fname in os.listdir(corpus_dir):
    if fname.endswith('.md'):
        loader = TextLoader(os.path.join(corpus_dir, fname), encoding='utf-8')
        docs = loader.load()
        for doc in docs:
            doc.metadata['source'] = fname
        documents.extend(docs)

print(f'Loaded {len(documents)} documents from corpus')

/var/folders/xn/gl_js0l969ncdrv1wc60st0m0000gn/T/ipykernel_66523/1568020257.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
/opt/anaconda3/envs/tutor_rag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 3 documents from corpus


## Step 2: Chunk the Documents

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = splitter.split_documents(documents)

print(f'Split into {len(chunks)} chunks')
print(f'Sample chunk:\n{chunks[0].page_content[:300]}')

Split into 18 chunks
Sample chunk:
# NLP Course Syllabus - CS 5340

## Course Overview
This course covers Natural Language Processing (NLP) theory and practice. Students will learn core NLP techniques, modern deep learning approaches, and how to build real-world NLP systems.

**Instructor:** Dr. Jane Smith  
**Credits:** 3  
**Prereq


## Step 3: Embed and Store in FAISS

Using `sentence-transformers/all-MiniLM-L6-v2` — free local embeddings, no API key needed.

FAISS Stores
Chunk Text
+
Vector
+
Metadata

Example:

{
  "text": "Python supports loops",
  "vector": [0.41, -0.22, ...],
  "source": "python.md"
}

mmr-Maximal Marginal Relevance

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_documents(chunks, embeddings)

retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 4, 'fetch_k': 10}
)
print('Vector store ready.')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5188.91it/s]


Vector store ready.


## Step 4: Build the Tutor RAG Chain (LCEL)

Using LangChain Expression Language (LCEL) — modern replacement for `RetrievalQA`.

In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

TUTOR_PROMPT = PromptTemplate(
    input_variables=['context', 'question'],
    template="""You are an expert NLP course tutor. Answer ONLY using the provided course materials below.
If the answer is not in the materials, say: This topic is not covered in the current course materials.

Be pedagogical: explain concepts clearly and reference specific modules or chapters.

Course Materials:
---------------
{context}
---------------

Student Question: {question}

Answer:"""
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

llm = ChatGroq(model='llama-3.1-8b-instant', temperature=0, api_key=GROQ_API_KEY)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | TUTOR_PROMPT
    | llm
    | StrOutputParser()
)

print('Tutor chain ready.')

Tutor chain ready.


## Step 5: Hallucination Detection

Semantic faithfulness — cosine similarity between answer and retrieved context embeddings.
- Score >= 0.75 → ✅ Answer is grounded
- Score < 0.75  → ⚠️ Potential hallucination

In [6]:
import numpy as np

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def check_hallucination(answer, source_docs, threshold=0.75):
    context = ' '.join([d.page_content for d in source_docs])
    answer_emb = embeddings.embed_query(answer)
    context_emb = embeddings.embed_query(context)
    score = cosine_similarity(answer_emb, context_emb)
    return {
        'faithfulness_score': round(score, 4),
        'is_hallucination': score < threshold
    }

print('Hallucination detector ready.')

Hallucination detector ready.


## Step 6: Ask Tutor Function

In [7]:
def ask_tutor(question):
    answer = chain.invoke(question)
    source_docs = retriever.invoke(question)
    result = check_hallucination(answer, source_docs)

    print(f"{'='*60}")
    print(f"Q: {question}\n")
    print(f"A: {answer}\n")
    verdict = '⚠️  POTENTIAL HALLUCINATION' if result['is_hallucination'] else '✅  Answer appears grounded'
    print(f"Faithfulness Score : {result['faithfulness_score']}")
    print(f"Verdict            : {verdict}")
    print(f"Sources            : {[d.metadata.get('source') for d in source_docs]}")
    print('='*60)
    return answer

## Step 7: Demo Questions

In [8]:
# Grounded question
ask_tutor('What is multi-head attention and why is it useful?')

Q: What is multi-head attention and why is it useful?

A: Multi-head attention is a mechanism that allows the model to attend to information from different representation subspaces. This is achieved by running multiple attention functions in parallel, each with its own set of linearly projected Query (Q), Key (K), and Value (V) matrices.

The formula for multi-head attention is given by:

```
MultiHead(Q,K,V) = Concat(head_1, ..., head_h) * W_O
where head_i = Attention(Q*W_Qi, K*W_Ki, V*W_Vi)
```

This allows the model to capture different aspects of the input data and weigh them differently, which can be useful for tasks such as machine translation, where the model needs to attend to different parts of the input sentence.

In Module 5: Transformers and Attention, we discussed the Transformer architecture, which uses multi-head attention as a key component. The Transformer architecture is designed to handle sequential data, such as text, and multi-head attention is a crucial part of th

'Multi-head attention is a mechanism that allows the model to attend to information from different representation subspaces. This is achieved by running multiple attention functions in parallel, each with its own set of linearly projected Query (Q), Key (K), and Value (V) matrices.\n\nThe formula for multi-head attention is given by:\n\n```\nMultiHead(Q,K,V) = Concat(head_1, ..., head_h) * W_O\nwhere head_i = Attention(Q*W_Qi, K*W_Ki, V*W_Vi)\n```\n\nThis allows the model to capture different aspects of the input data and weigh them differently, which can be useful for tasks such as machine translation, where the model needs to attend to different parts of the input sentence.\n\nIn Module 5: Transformers and Attention, we discussed the Transformer architecture, which uses multi-head attention as a key component. The Transformer architecture is designed to handle sequential data, such as text, and multi-head attention is a crucial part of this architecture.\n\nTo summarize, multi-head a

In [9]:
# RAG pipeline question
ask_tutor('What are the stages of a RAG pipeline and what is HyDE?')

Q: What are the stages of a RAG pipeline and what is HyDE?

A: This topic is not covered in the current course materials.

Faithfulness Score : 0.0946
Verdict            : ⚠️  POTENTIAL HALLUCINATION
Sources            : ['chapter7_rag.md', 'chapter5_transformers.md', 'chapter5_transformers.md', 'chapter7_rag.md']


'This topic is not covered in the current course materials.'

In [10]:
# Out-of-domain — should say not in materials
ask_tutor('What is the derivative of sin(x)?')

Q: What is the derivative of sin(x)?

A: This topic is not covered in the current course materials.

Faithfulness Score : 0.2085
Verdict            : ⚠️  POTENTIAL HALLUCINATION
Sources            : ['nlp_course_syllabus.md', 'chapter7_rag.md', 'nlp_course_syllabus.md', 'chapter7_rag.md']


'This topic is not covered in the current course materials.'

## Step 8: Batch Hallucination Evaluation Report

In [11]:
import pandas as pd

eval_questions = [
    'What is the difference between BERT and GPT?',
    'Explain faithfulness metric in RAG evaluation.',
    'What is positional encoding in Transformers?',
    'What is the grading breakdown for this NLP course?',
    'How does quantum computing work?',  # out-of-domain
]

rows = []
for q in eval_questions:
    answer = chain.invoke(q)
    source_docs = retriever.invoke(q)
    result = check_hallucination(answer, source_docs)
    rows.append({
        'Question': q[:60],
        'Faithfulness Score': result['faithfulness_score'],
        'Hallucination': '⚠️ Yes' if result['is_hallucination'] else '✅ No',
        'Sources': ', '.join(set(d.metadata.get('source') for d in source_docs))
    })

pd.DataFrame(rows)

,Question,Faithfulness Score,Hallucination,Sources
0,What is the difference between BERT and GPT?,0.7246,⚠️ Yes,"nlp_course_syllabus.md, chapter5_transformers.md"
1,Explain faithfulness metric in RAG evaluation.,0.7358,⚠️ Yes,"nlp_course_syllabus.md, chapter7_rag.md"
2,What is positional encoding in Transformers?,0.7173,⚠️ Yes,"chapter5_transformers.md, chapter7_rag.md"
3,What is the grading breakdown for this NLP cou...,0.4887,⚠️ Yes,"nlp_course_syllabus.md, chapter5_transformers...."
4,How does quantum computing work?,0.0292,⚠️ Yes,"chapter5_transformers.md, chapter7_rag.md"


## Step 9: Interactive Chat Loop

In [12]:
print("NLP Tutor Chatbot — type 'exit' to quit\n")
while True:
    question = input('You: ').strip()
    if question.lower() in ('exit', 'quit', ''):
        print('Goodbye!')
        break
    ask_tutor(question)

NLP Tutor Chatbot — type 'exit' to quit

Q: steps of rag

A: According to Chapter 7: Retrieval-Augmented Generation (RAG), the standard RAG pipeline has five stages:

1. **Document Loading**: Load raw documents (PDFs, markdown, web pages) using document loaders.
2. **Chunking**: Split documents into smaller chunks (typically 512–1024 tokens) with overlap (100–200 tokens) to preserve context at boundaries.
3. **Query Expansion**: Generate multiple query variants and retrieve for each, then merge results.
4. **Parent-Child Chunking**: Store small chunks for retrieval precision but return larger parent chunks for generation context.

These stages are crucial in the RAG pipeline, enabling the model to effectively retrieve and utilize relevant context for generation.

Faithfulness Score : 0.86
Verdict            : ✅  Answer appears grounded
Sources            : ['chapter7_rag.md', 'chapter7_rag.md', 'nlp_course_syllabus.md', 'chapter5_transformers.md']
Goodbye!
